# 3. Pipeline Spark e Delta Lake

**Equipe:** Diogo Galrão Carvalho; Felipe Artur Macedo Lima; Luan Cavalcante Dias Rodrigues.

**Disciplina:** Banco de Dados e Big Data para Data Science, Pós-Graduação em Data Science e Analytics.

**Execução local em Docker, Spark 3.5.3, Java 17, Delta 3.2.0.** O experimento não foi executado no Databricks. O conector lê o MongoDB local e a demografia vem do CSV. A implementação compartilhada está em `scripts/spark_pipeline.py`.

In [1]:
from pathlib import Path
import sys
import os
ROOT = Path.cwd() if (Path.cwd() / 'scripts').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

import pandas as pd
from scripts.spark_pipeline import create_spark, run
spark = create_spark()
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version, "Master:", spark.sparkContext.master)

/usr/local/lib/python3.12/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found


https://repo.maven.apache.org/maven2 added as a remote repository with the name: repo-1
Ivy Default Cache set to: /home/estudante/.ivy2/cache
The jars for the packages stored in: /home/estudante/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bb8bc3d6-62ce-4c0e-a772-80ca739345cf;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central


:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.mongodb.spark#mongo-spark-connector_2.12;10.4.0 in central


	found org.mongodb#mongodb-driver-sync;5.1.4 in central
	[5.1.4] org.mongodb#mongodb-driver-sync;[5.1.1,5.1.99)
	found org.mongodb#bson;5.1.4 in central
	found org.mongodb#mongodb-driver-core;5.1.4 in central
	found org.mongodb#bson-record-codec;5.1.4 in central
:: resolution report :: resolve 2787ms :: artifacts dl 14ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.mongodb#bson;5.1.4 from central in [default]
	org.mongodb#bson-record-codec;5.1.4 from central in [default]
	org.mongodb#mongodb-driver-core;5.1.4 from central in [default]
	org.mongodb#mongodb-driver-sync;5.1.4 from central in [default]
	org.mongodb.spark#mongo-spark-connector_2.12;10.4.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       

26/09/29 00:05:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark: 3.5.3 Master: local[2]


## Ingestão, transformações e validação

O fluxo lê `jogadores` via MongoDB Spark Connector; aplica `explode`; verifica duplicidade e cobertura demográfica; faz broadcast join; calcula métricas e grava Delta. Compras são comparadas campo a campo com PostgreSQL. A releitura Delta é comparada com `exceptAll` nas duas direções. Uma falha na leitura do MongoDB interrompe a execução, sem substituir a origem por JSON.

In [2]:
try:
    result = run(spark, repetitions=5)
finally:
    spark.stop()
print("Origem:", result["source"])
print("Reconciliação PostgreSQL:", result["relational_reconciliation"])
print("Linhas Delta:", result["delta_rows"])
print("Reconciliação Delta:", result["delta_reconciliation"])

Origem: mongodb_connector
Reconciliação PostgreSQL: all_purchase_fields_equal
Linhas Delta: 1043
Reconciliação Delta: exceptAll_both_directions_empty


## Métricas da simulação

Ticket médio é ouro por transação. Ouro não é receita monetária. Rankings por quantidade e por valor respondem perguntas distintas. Dias da semana seguem Spark: domingo=1 e sábado=7.

In [3]:
m = result["metrics"]
print("Compras:", m["purchases"], "Ouro:", m["gold"], "Fim de semana (%):", m["weekend_percent"])
display(pd.DataFrame(m["region_elo"]))
display(pd.DataFrame(m["top_volume"]))
display(pd.DataFrame(m["top_gold"]))

Compras: 1043 Ouro: 2228821 Fim de semana (%): 30.8904


,regiao,elo,total_ouro,unidades,transacoes,ticket_medio_ouro
0,KR,Desafiante,351612,193,170,2068.31
1,BR,Mestre,335025,166,144,2326.56
2,BR,Diamante,330861,180,156,2120.90
3,KR,Grão-Mestre,318825,163,136,2344.30
4,KR,Mestre,227783,123,107,2128.81
5,BR,Esmeralda,162374,90,82,1980.17
6,BR,Platina,160800,99,89,1806.74
7,BR,Ouro,116283,67,59,1970.90
8,EUW,Mestre,90350,50,43,2101.16
9,BR,Grão-Mestre,78000,42,31,2516.13


,id_item,item,unidades,total_ouro,transacoes
0,1082,Lacre Sombrio,14,4900,13
1,1105,Broto de Esmagamusgo,11,4950,10
2,2138,Elixir de Ferro,11,5500,11
3,2522,Concretizador,11,30800,9
4,3165,Morellonomicon,11,31350,9
5,3870,Criassonhos,11,4400,9
6,1120,Elmo de Doran,10,4500,7
7,3020,Sapatos do Feiticeiro,10,11000,8
8,3599,Lança Negra da Kalista,10,1500,8
9,4401,Força da Natureza,10,28000,7


,id_item,item,unidades,total_ouro,transacoes
0,3165,Morellonomicon,11,31350,9
1,2522,Concretizador,11,30800,9
2,4401,Força da Natureza,10,28000,7
3,3508,Colhedor de Essência,9,27450,8
4,3071,Cutelo Negro,9,27000,6
5,3073,Hexoplaca Experimental,9,27000,7
6,3087,Faca de Statikk,9,27000,6
7,663193,Placa Gargolítica,10,25000,10
8,3083,Armadura de Warmog,8,24800,6
9,3084,Coração de Aço,8,24000,6


In [4]:
display(pd.DataFrame(m["weekday_platform"]))
display(pd.DataFrame(m["subscriptions"]))

,dia_semana,plataforma,total_ouro,transacoes
0,1,Desktop Gamer,354333,154
1,1,Notebook Gamer,35100,19
2,2,Desktop Gamer,274275,124
3,2,Notebook Gamer,54100,27
4,3,Desktop Gamer,309290,148
5,3,Notebook Gamer,34850,20
6,4,Desktop Gamer,173250,92
7,4,Notebook Gamer,35250,17
8,5,Desktop Gamer,321041,139
9,5,Notebook Gamer,47200,24


,tier_assinatura,total_ouro,transacoes,ticket_medio_ouro
0,VIP,1185024,518,2287.69
1,Gratuito,430520,224,1921.96
2,Pro,275300,127,2167.72


## Experimento de desempenho

O experimento mede cinco repetições após o aquecimento e verifica a igualdade dos resultados. Compara SortMergeJoin e BroadcastHashJoin na mesma agregação por região/elo, alternando a ordem das execuções e desativando AQE e broadcast automático. A comparação com e sem cache usa a mesma consulta e registra a materialização separadamente. Os planos e todas as amostras estão em `reports/evidence`. Medições sequenciais de cache ainda podem sofrer efeitos de aquecimento; não demonstram ganhos em clusters distribuídos.

In [5]:
b = result["benchmark"]
display(pd.DataFrame(b["samples_seconds"]))
print("Medianas (s):", b["median_seconds"])
print("Materialização cache (s):", b["cache_materialization_seconds"])
print("Razão sem/com cache:", b["cache_speedup"])
print("Razão sort-merge/broadcast:", b["join_speedup"])

,sort_merge,broadcast,uncached,cached
0,0.328877,0.271510,0.262948,0.141414
1,0.287043,0.258913,0.231440,0.154836
2,0.272794,0.268514,0.252592,0.164746
3,0.338211,0.280233,0.207613,0.197626
4,0.272439,0.236607,0.230016,0.153068


Medianas (s): {'sort_merge': 0.28704291099711554, 'broadcast': 0.26851373400131706, 'uncached': 0.23144026700174436, 'cached': 0.1548356600033003}
Materialização cache (s): 0.3386609769950155
Razão sem/com cache: 1.4947478313252338
Razão sort-merge/broadcast: 1.0690064404515993


## Conclusão

A execução com a base pequena verificou a integração e a conservação dos dados. A escalabilidade ainda precisa ser medida. Broadcast ainda transmite a dimensão; agregações podem fazer shuffle. Particionar esta pequena saída Delta por região não foi necessário. Consulte os relatórios para valores medidos e limitações, incluindo a adaptação do requisito de nuvem.